# الدرس 09 - نمط تصميم التفكير فوق المعرفي


## الإعداد

يعرض هذا الدفتر نمط تصميم الإدراك العالي باستخدام إطار عمل وكيل مايكروسوفت.

**المتطلبات الأساسية:**
- إعداد نشر Azure OpenAI عبر متغيرات البيئة
- تم المصادقة على Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ما هو التَّفَكُّر في التَّفكير؟

التَّفَكُّر في التَّفكير هو **التفكير حول التفكير**. في سياق وكلاء الذكاء الاصطناعي، يعني بناء وكلاء يمكنهم:

- **التفكير الذاتي** في مخرجاتهم وعملية الاستدلال الخاصة بهم
- **الكشف عن الأخطاء** والتعافي بسلاسة بدلاً من الفشل بصمت
- **تقييم** ما إذا كانت ردودهم كاملة ومفيدة
- **تكييف** استراتيجيتهم عندما لا تنجح الطريقة الأولية (مثل الرجوع إلى نظام احتياطي)

الوكيل التَّفَكري لا يجيب فقط على الأسئلة — بل يراقب أدائه الخاص ويضبط نفسه في وقت التشغيل.


## الأدوات الأساسية والاحتياطية

نمط شائع في التفكير فوق المعرفي هو **استراتيجية التراجع**. يحاول الوكيل استخدام الأداة الأساسية أولاً؛ إذا فشل (مثلاً، خطأ 404)، يتعرف الوكيل على الفشل وينتقل بشفافية إلى أداة احتياطية.

هذا يعكس أنظمة العالم الحقيقي حيث قد تكون الخدمات الأساسية غير متاحة ويجب على الوكلاء تشخيص المشكلة ذاتياً قبل اختيار مسار بديل.

فيما يلي نحدد أداتين للبحث عن الرحلات الجوية:
- **الأداة الأساسية** — تغطي باريس، طوكيو، وبرشلونة
- **الأداة الاحتياطية** — تغطي برلين، سيدني، ومدينة نيويورك


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## وكيل يعكس نفسه مع استرداد الأخطاء

الوكيل أدناه مُوجه لمحاولة استخدام نظام الطيران الأساسي أولاً، التعرف على الإخفاقات، والرجوع بشفافية إلى نظام النسخ الاحتياطي. بعد كل رد، يقوم بتقييم ذاتي موجز لما إذا كان قد أجاب بالكامل على سؤال المستخدم.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## نمط التقييم الذاتي

جانب آخر من الميتامعرفية هو **التقييم الذاتي**: وكيل منفصل (أو نفس الوكيل في تمريرة ثانية) يراجع الرد من حيث الشمول والدقة والفائدة.

أدناه ننشئ وكيل `ResponseEvaluator` الذي يقوم بتقييم ردود وكلاء السفر على ثلاثة أبعاد.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## ملخص

في هذا الدرس تعلمت كيف تبني **وكلاء ما وراء المعرفة** باستخدام إطار عمل Microsoft Agent:

- **التأمل الذاتي**: وكلاء يراقبون تفكيرهم الخاص ويتواصلون بشفافية عما حدث.
- **استرداد الأخطاء مع البدائل**: نمط أداة رئيسية + احتياطية حيث يكتشف الوكيل الفشل (مثل أخطاء 404) ويحاول تلقائياً مصدرًا بديلاً.
- **التقييم الذاتي**: وكيل مقيم منفصل يقيم الإجابات من حيث الاكتمال والدقة والفائدة.

تجعل هذه الأنماط الوكلاء أكثر متانة وشفافية وجديرة بالثقة - وهي صفات حاسمة لنشرها في بيئات الإنتاج.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
